## Importing Libraries

In [1]:
from ollama import chat
import glob
from tqdm import tqdm
import os
import json
import re
import unicodedata
from groq import Groq
from difflib import SequenceMatcher

## Setting up files

In [ ]:
GENERATION_MODEL = "qwen3:8b" 
GROQ_MODEL = "openai/gpt-oss-120b"

GROQ_KEY = os.getenv("GROQ_API_KEY")
CLIENT = Groq(api_key=GROQ_KEY)

TYPE_LLM = True # True - local, False - groq

FILES_EXTR = glob.glob("../Test_Files/Clinical_trials/clinical-trial_*.txt")
GOLD_FILES = glob.glob("../Test_Files/Clinical_trials/GT-clinical-trial_*.json")

PROMPT_EXTR_FILE = "./prompts/criteria_extraction/criteria-extraction_prompt.txt"
SYS_PROMPT_EXTR_FILE = "./prompts/criteria_extraction/sys_criteria-extraction_prompt.txt"

OUTPUT_EXTR_DIR = "./llm-outputs/criteria-extraction/"
OUTPUT_EXTR_FILE = "experiment"

print(f"Found the following files for extraction - {FILES_EXTR}")
print(f"Found the following golden diaries {GOLD_FILES}")

Found the following files for extraction - ['../Test_Files/Clinical_trials\\clinical-trial_e1.txt', '../Test_Files/Clinical_trials\\clinical-trial_e10.txt', '../Test_Files/Clinical_trials\\clinical-trial_e11.txt', '../Test_Files/Clinical_trials\\clinical-trial_e2.txt', '../Test_Files/Clinical_trials\\clinical-trial_e3.txt', '../Test_Files/Clinical_trials\\clinical-trial_e4.txt', '../Test_Files/Clinical_trials\\clinical-trial_e5.txt', '../Test_Files/Clinical_trials\\clinical-trial_e6.txt', '../Test_Files/Clinical_trials\\clinical-trial_e7.txt', '../Test_Files/Clinical_trials\\clinical-trial_e8.txt', '../Test_Files/Clinical_trials\\clinical-trial_e9.txt']
Found the following golden diaries ['../Test_Files/Clinical_trials\\GT-clinical-trial_e1.json', '../Test_Files/Clinical_trials\\GT-clinical-trial_e10.json', '../Test_Files/Clinical_trials\\GT-clinical-trial_e11.json', '../Test_Files/Clinical_trials\\GT-clinical-trial_e2.json', '../Test_Files/Clinical_trials\\GT-clinical-trial_e3.json', 

## Pre-processing

In [3]:
def normalize_docs(text):
    text = normalize_text(text)

    inclusion_match = re.search(
        r"(Inclusion Criteria\s*:?\s*)(.*?)(?=Exclusion Criteria\s*:?)",
        text,
        re.IGNORECASE | re.DOTALL,
    )

    exclusion_match = re.search(
        r"(Exclusion Criteria\s*:?\s*)(.*?)(?=\n(?:Study Plan|Study Design|Investigational Product|Control Product|Study Endpoints|Primary Endpoint|Secondary Endpoints|Safety Endpoints|Follow-Up|Statistical Analysis|References)\b|\Z)",
        text,
        re.IGNORECASE | re.DOTALL,
    )

    if inclusion_match and exclusion_match:

        inclusion_text = inclusion_match.group(2).strip()
        exclusion_text = exclusion_match.group(2).strip()

        text = (
            "Inclusion Criteria:\n"
            f"{inclusion_text}\n\n"
            "Exclusion Criteria:\n"
            f"{exclusion_text}"
        )

    print(f"[DEBUG] Normalized Trial: {text}")

    return text

def normalize_text(text):
    text = unicodedata.normalize("NFKC", text)
    
    text = re.sub(r"[‐-‒–—]", "-", text)

    text = re.sub(r"[ \t]+", " ", text)

    text = re.sub(r"\r\n?", "\n", text)

    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

## Setting up environment

In [4]:
## Setting evironment
def set_env(prompt_file, sys_prompt_file, output_dir):
    with open(prompt_file,"r", encoding="utf-8") as p:
        base_prompt = p.read().strip()
        
    with open(sys_prompt_file, "r", encoding="utf-8") as sp:
        sys_prompt = sp.read().strip()

    os.makedirs(output_dir,exist_ok=True)

    count = 0

    for path in os.listdir(output_dir):
        if os.path.isfile(os.path.join(output_dir, path)):
            count += 1
    
    return base_prompt, sys_prompt, count

base_prompt_extr, sys_prompt_extr, count_extr_exp = set_env(PROMPT_EXTR_FILE, SYS_PROMPT_EXTR_FILE, OUTPUT_EXTR_DIR)

## Criteria Extraction
In this first phase the criteria of a given clinical trial are extracted still in natural language to make the conversion easier

In [5]:
pbar = tqdm(total=len(FILES_EXTR), desc="Processing trials for criteria extraction")

for file in FILES_EXTR:
    with open(file,"r", encoding="utf-8") as f:
        text = f.read()
        
        normalized_text = normalize_docs(text)
        
        print(f"processing file: {file}")
        
        prompt = base_prompt_extr.replace("{{TRIAL_TEXT}}", normalized_text)
        
        print(f"System prompt for file {file}:\n{sys_prompt_extr}\n")
        print(f"Prompt for file {file}:\n{prompt}\n")
        
        if TYPE_LLM:
            stream = chat(
                model=GENERATION_MODEL,
                messages=[
                    {
                        "role": "system",
                        "content": sys_prompt_extr
                    },
                    {
                        "role": "user", 
                        "content": prompt
                        }
                    ],
                stream=True,
                options={"num_ctx": 32000}
                )
            
            llm_output = ""
            for chunk in stream:
                llm_output += chunk["message"]["content"]
                
        elif not TYPE_LLM:
            stream = CLIENT.chat.completions.create(
                model= GROQ_MODEL,
                messages=[
                    {
                        "role": "system",
                        "content": sys_prompt_extr
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=0
            )
            
            llm_output = stream.choices[0].message.content

        with open(f"{OUTPUT_EXTR_DIR}{OUTPUT_EXTR_FILE}-{count_extr_exp}.txt","a",encoding="utf-8") as o:
            o.write(f"Ouput for file {file}\n")
            o.write(f"{llm_output}\n\n")
            print(f"Saved LLM output on {OUTPUT_EXTR_FILE}-{count_extr_exp}")
            
        
        print("\n")
        
        pbar.update(1)
        
pbar.close()

Processing trials for criteria extraction:   0%|          | 0/11 [00:00<?, ?it/s]

[DEBUG] Normalized Trial: Inclusion Criteria:
1. Age ≥ 18 years.
2. Histologically or cytologically confirmed metastatic NSCLC (Stage IV).
3. Documented progression after first-line platinum-based chemotherapy combined with anti-PD-1 or anti-PD-L1 therapy.
4. ECOG Performance Status 0-1.
5. At least one measurable lesion per RECIST 1.1.
6. Adequate organ function:
 - Absolute neutrophil count (ANC) ≥ 1.5 x 10^9/L
 - Platelets ≥ 100 x 10^9/L
 - Hemoglobin ≥ 9 g/dL
 - AST and ALT ≤ 2.5 x ULN (≤ 5 x ULN if liver metastases)
 - Total bilirubin ≤ 1.5 x ULN
 - Creatinine clearance ≥ 40 mL/min (CKD-EPI formula)
7. Women of childbearing potential must have a negative pregnancy test prior to treatment initiation.
8. Signed informed consent prior to any study-specific procedure.

Exclusion Criteria:
1. Known EGFR, ALK, or ROS1 genomic alterations with available approved targeted therapy.
2. Untreated or symptomatic brain metastases.
3. Active autoimmune disease requiring systemic immunosuppressi

Processing trials for criteria extraction:   9%|▉         | 1/11 [06:50<1:08:28, 410.88s/it]

Saved LLM output on experiment-2


[DEBUG] Normalized Trial: Study Overview
Brief Summary
This is an open, multicenter phase II trial of therapy with a combination of cetuximab, and irinotecan every second week combined with a daily dose of everolimus to patients with metastatic colorectal cancer with Kirsten rat sarcoma viral oncogene (KRAS) mutation or to patients resistent to cetuximab and irinotecan therapy for metastatic colorectal cancer.
Detailed Description
Main objective:

Number of patients with progressive disease that obtain disease control defined as the sum of patients that obtain a Complete Remission (CR), Partial Remission(PR, or stable disease (SD))
Secondary objectives:

Time to progression after first therapy.
Length of disease control (CR, PR and SD)
Survival from date of start of therapy.
Safety and toxicity of the therapy graded according to Common Toxicity Criteria version 3.0
Influence of smoking on disease control, response, survival and time to progression and

Processing trials for criteria extraction:  18%|█▊        | 2/11 [13:30<1:00:40, 404.49s/it]

...

Saved LLM output on experiment-2


[DEBUG] Normalized Trial: Inclusion Criteria:
Participants must be >18 years old at time of diagnosis
Histologically confirmed non-small cell lung cancer
ECOG PS 2
Clinical staging of IIIc or IV disease.
4A. For patients with stage IIIc disease, patients are ineligible for or refuse standard treatment with platinum-double chemotherapy and radiation.

4B. For patients with stage IV disease, platinum doublet chemotherapy is not appropriate, deemed unsafe by the treating physician, or declined by the patient

4C. Screening lab work must meet the following parameters:

4Ca. Absolute neutrophil count (ANC) ≥1000/mm3

4Cb. Platelet count ≥100,000/mm3

4Cc. CrCl>50 (if pemetrexed is to be offered)

4Cd. AST and ALT ≤ 2.5 x ULN

4D. Patients with small, asymptomatic brain metastases are eligible

4E. Women of childbearing potential must be negative for pregnancy testing (urine or blood) and agree to use effective contraception. Viable contraception should be 

Processing trials for criteria extraction:  91%|█████████ | 10/11 [1:42:34<09:07, 547.04s/it]

Saved LLM output on experiment-2


[DEBUG] Normalized Trial: Inclusion Criteria:
>= 18 years of age Have an Eastern Cooperative Oncology Group (ECOG) performance status of =< 1 at the time of study treatment initiation Histologically or cytologically confirmed diagnosis of small cell lung cancer (SCLC) Patient should have extensive stage disease, defined as, malignant pleural effusion, pulmonary metastases in the contralateral lung, and/or the presence of extra-thoracic metastatic disease Must have measurable disease based on Response Evaluation Criteria in Solid Tumors (RECIST) 1.1 prior to starting platinum-based systemic chemotherapy Absolute neutrophil count (ANC) >= 1.5 x 10^9/L Platelets >= 100 x 10^9/L Hemoglobin >= 9 g/dL Serum creatinine =< 1.5 x institution upper limit of normal (ULN) and calculated creatinine clearance of at least 15 ml/min.

Alanine aminotransferase (ALT) and aspartate aminotransferase (AST) =< 3 x upper limit of normal (ULN) (ALT and AST =< 5 x ULN is acce

Processing trials for criteria extraction: 100%|██████████| 11/11 [1:46:09<00:00, 579.01s/it]

Saved LLM output on experiment-2


